# Day 14 — Full OOP Refactor

> ⚠️ **Why this matters.** Today is mostly refactoring — no new Python concepts. You take everything from Days 11-13 and put it together. Result: english-helper looks like real software. Code review-able. Extensible. Tested-friendly (which matters Day 16+).

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/jakkzz/prince-curriculum/blob/main/phase-1-python-cli/lessons/14-oop-refactor.ipynb)

## What you'll do today

**Time:** 30 min reading + 2.5 hours refactor + 30 min quiz.

By the end:

- [ ] `Word` (dataclass), `WordStore` (class), `QuizMode` (ABC) all in place
- [ ] `cli.py` uses these instead of dicts
- [ ] All tests (manual) still pass
- [ ] You can describe in 2 minutes why this is better than the dict version

## The target architecture

```mermaid
graph TB
    CLI[cli.py: REPL] --> Store[WordStore class<br/>storage.py]
    CLI --> API[api.py: fetch_word]
    CLI --> Quizzer[Quizzer class<br/>quiz.py]
    Store --> Word[Word dataclass<br/>word.py]
    Quizzer --> Word
    Quizzer --> Modes[QuizMode ABC<br/>quiz_modes.py]
```

**Five files, five clear responsibilities.** That's the standard you'll uphold in every project from here on.

## The classes

### `Word` (Day 12)

```python
@dataclass(frozen=True)
class Word:
    word: str
    ipa: str = ''
    thai: str = ''
    definition: str = ''

    @property
    def display(self) -> str: ...
    def to_dict(self) -> dict: ...
    @classmethod
    def from_dict(cls, d: dict) -> 'Word': ...
```

### `WordStore` (new today)

```python
@dataclass
class WordStore:
    words: dict[str, Word] = field(default_factory=dict)
    path: Path = field(default_factory=lambda: Path.home() / '.english-helper' / 'words.json')

    def add(self, w: Word) -> None: ...
    def remove(self, word: str) -> bool: ...
    def lookup(self, word: str) -> Word | None: ...
    def all(self) -> list[Word]: ...
    def save(self) -> None: ...
    @classmethod
    def load(cls, path: Path | None = None) -> 'WordStore': ...
```

### `Quizzer` (composes a QuizMode)

```python
@dataclass
class Quizzer:
    store: WordStore
    mode: QuizMode

    def run(self, rounds: int = 5) -> tuple[int, int]: ...
```

## End-of-day mini-project — full refactor

> 🎯 **Today is the refactor.** No new feature. Same behavior, new structure.

### Steps

1. Move all code into the structure shown above. Five files in `src/english_helper/`.
2. Update `cli.py` commands to use these classes:
   ```python
   def cmd_add(store: WordStore, args: list[str]) -> str:
       if not args: return 'Usage: add WORD'
       word = args[0].lower()
       if store.lookup(word): return f'{word!r} already in vocab'
       ...
       store.add(Word(word=word, ipa=ipa, definition=defn))
       return f'Added {word!r}'
   ```
3. Run mypy on the whole tree. Zero errors.
4. Run ruff. Zero warnings.
5. Manually test the acceptance script from Day 10. **Same input → same output.**

### Code review

On the Friday call you'll walk the mentor through: "this is `cli.py`. This line calls `store.add(...)` which is in `storage.py` and looks like this...". If you can do this for every command, you've succeeded.

## Connect to the project

> 🎯 **Tomorrow (Day 15):** spaced repetition scheduler — the algorithm dictionary apps like Anki use. With clean classes, this is one new file (`scheduler.py`).

**Quiz:** [14-oop-refactor-quiz.ipynb](14-oop-refactor-quiz.ipynb)